In [1]:
from matplotlib.colors import LinearSegmentedColormap
from captum.attr import visualization as viz
import matplotlib.pyplot as plt

import os
import numpy as np
import pandas as pd
import time
from datetime import datetime 
import pdb
import yaml
import sys
import glob
import shutil
import tqdm
import imageio

In [74]:
listo = []

for seed in range(10):
    path = f'/gpfs/data/mayerhoeferlab/projects/myeloma/results/2D_all_seeds_prediction/test/pred_test_{seed}.csv'
    xx = pd.read_csv(path)
    xx['model_number'] = seed
    listo.append(xx)
probs_2D = pd.concat(listo, axis=0).reset_index(drop=True)

listo = []
for seed in range(10):
    path = f'/gpfs/data/mayerhoeferlab/projects/myeloma/results/2D_dual_mipview_all_seeds_prediction/test/pred_test_{seed}.csv'
    xx = pd.read_csv(path)
    xx['model_number'] = seed
    listo.append(xx)
probs_dual_mipview = pd.concat(listo, axis=0).reset_index(drop=True)

probs_2D = probs_2D[['accession', 'mipview', 'probs', 'model_number']]
probs_dual_mipview = probs_dual_mipview[['accession', 'probs', 'model_number']]

# groupby and summarise
probs_2D = probs_2D.groupby(['accession', 'mipview']).agg(probs_mean = ('probs', 'mean'), probs_range = ('probs', lambda x: (round(min(x),2), round(max(x),2))), probs_concat = ('probs', lambda x: sorted(list(round(x, 2))))).reset_index()
probs_dual_mipview = probs_dual_mipview.groupby(['accession']).agg(probs_mean = ('probs', 'mean'), probs_range = ('probs', lambda x: (round(min(x),2), round(max(x),2))), probs_concat = ('probs', lambda x: sorted(list(round(x, 2))))).reset_index()

In [71]:
path = '/gpfs/data/mayerhoeferlab/projects/myeloma/datafiles_for_mars/other/MARS-FPs-final.xlsx'

df_2D = pd.read_excel(path, sheet_name='2d-all')
df_2D = df_2D.drop('1=Incorrect labeling, 2=Infect/inflam/degen, 3=Iatrogenic, 4=Brown fat, 5=Other, 6=Unknown', axis=1)
df_dual_mipview = pd.read_excel(path, sheet_name='2d-dual-mipview')

In [75]:
res = df_2D.merge(probs_2D[['accession', 'mipview', 'probs_mean', 'probs_range', 'probs_concat']], left_on='Acc', right_on='accession', how='left')
res = res.drop('accession', axis=1)
res.to_excel('/gpfs/data/mayerhoeferlab/projects/myeloma/datafiles_for_mars/other/MARS-FPs-final-with-model-probs-2D.xlsx', index=False)